# Public API access points and live retrieval checks

Start the backend from this repository:

```sh
uv run uvicorn API.server:app --reload --reload-dir API --port 8002
```

Run all cells to test the HTTP contract, all 20 source adapters, location metadata and actual evidence downloads. Public portals can block access or have limited municipal coverage. The notebook reports these separately from HTTP/schema failures, which always raise errors. No API keys or customer information are needed.

In [1]:
import hashlib
import json
import os
from pathlib import Path
from time import perf_counter

import polars as pl
import requests
from IPython.display import display

BASE_URL = os.getenv("SITE_API_URL", "http://127.0.0.1:8002").rstrip("/")
POINT = {"latitude": 42.359, "longitude": -71.093}
OUTPUT = Path("data/processed/api_demo")
OUTPUT.mkdir(parents=True, exist_ok=True)
REQUIRE_ALL_PROVIDERS = False  # Set True only when requiring complete external provider coverage.
session = requests.Session()
checks = []


def get(path, params=None, expected=200):
    started = perf_counter()
    try:
        response = session.get(BASE_URL + path, params=params, timeout=360)
    except requests.ConnectionError as error:
        raise RuntimeError(
            f"Cannot connect to {BASE_URL}. Start the backend from the repository root: "
            "uv run uvicorn API.server:app --reload --port 8002"
        ) from error
    checks.append({"path": path, "url": response.url, "expected_status": expected,
                   "status": response.status_code, "passed": response.status_code == expected,
                   "seconds": round(perf_counter() - started, 3)})
    if response.status_code != expected:
        raise RuntimeError(f"GET {response.url}: expected HTTP {expected}, got {response.status_code}. {response.text[:500]}")
    return response


## Check the running API before retrieving data

The API advertises its schema and table columns in `/sources`. This check detects an old process before later table assertions can fail. No local backend imports are used, so a previously imported Python module cannot set the expected columns.


In [2]:
assert get("/health").json() == {"status": "ok"}
schema = get("/openapi.json").json()
catalog = get("/sources").json()
required_columns = {
    "checklist item", "subcategory_name", "subcategory_value", "data_source", "source_title",
    "fetched_at_utc", "source_date", "source_date_basis", "local_file", "file_url",
    "searched_latitude", "searched_longitude", "source_latitude", "source_longitude",
    "source_distance_m", "source_location_basis", "match_method",
}
server_columns = catalog.get("table_columns", [])
missing_columns = required_columns - set(server_columns)
api_version = schema.get("info", {}).get("version", "unknown")
if catalog.get("schema_version") != "1.3" or missing_columns:
    raise RuntimeError(
        f"Incompatible API at {BASE_URL}: version {api_version}, "
        f"schema {catalog.get('schema_version', 'not advertised')}. "
        f"Missing table columns: {sorted(missing_columns)}. "
        "An older API is still running on port 8001. Use http://127.0.0.1:8002 "
        "and start the current backend with: uv run uvicorn API.server:app --reload --port 8002. "
        "Check SITE_API_URL if this notebook still points to the old server."
    )
expected_sources = {"utilities", "parcels", "buildings", "permits", "lber", "flood", "elevation"}
assert expected_sources <= set(catalog["automated"]), f"Unexpected provider catalog at {BASE_URL}: {list(catalog['automated'])}"
assert {"/site", "/site/table", "/evidence/{file_path}"} <= set(schema["paths"])
get("/docs")
get("/redoc")
print(f"Connected to {BASE_URL} — API {api_version}, schema {catalog['schema_version']}, {len(server_columns)} table columns")
display(pl.from_dicts([{"source": name, "title": value["title"], "match_method": value["match_method"]} for name, value in catalog["automated"].items()]))


Connected to http://127.0.0.1:8002 — API 1.4.0, schema 1.3, 17 table columns


source,title,match_method
str,str,str
"""utilities""","""MassGIS — electricity provider…","""municipality_containing_point"""
"""parcels""","""MassGIS — Massachusetts assess…","""point_intersection"""
"""buildings""","""MassGIS — building structures …","""point_intersection"""
"""lber""","""Massachusetts DOER — multi-yea…","""within_250m_candidate"""
"""flood""","""FEMA — National Flood Hazard L…","""point_intersection"""
…,…,…
"""environmental""","""MassDEP Tier Classified Chapte…","""parcel_buffer_candidate"""
"""activity_use_limits""","""MassDEP Activity and Use Limit…","""parcel_buffer_candidate"""
"""protected_habitat""","""MassGIS NHESP priority habitat""","""parcel_intersection"""


## Full public lookup and table

The JSON includes `public_coverage`, explicit missing public/engineering facts and downloadable evidence. `download_evidence=False` skips the exported evidence package; required source files (for example PTS spreadsheets and tariff PDFs) are still cached for extraction.

In [3]:
site = get("/site", POINT).json()
assert site["schema_version"] == "1.3"
assert set(site["sources"]) == set(catalog["default_sources"])
(OUTPUT / "site.json").write_text(json.dumps(site, indent=2))
rows = get("/site/table", {**POINT, "download_evidence": False}).json()
if not isinstance(rows, list) or not rows:
    raise RuntimeError(f"Expected non-empty table rows from {BASE_URL}/site/table; received {str(rows)[:300]}")
missing = [(index, sorted(set(server_columns) - row.keys())) for index, row in enumerate(rows) if set(server_columns) - row.keys()]
if missing:
    raise RuntimeError(f"Table does not match the advertised schema at {BASE_URL}. First incomplete row: {missing[0]}. Restart the API and rerun from the top.")
table = pl.from_dicts(rows, infer_schema_length=None).select(server_columns)
assert table["searched_latitude"].eq(POINT["latitude"]).all()
assert table["searched_longitude"].eq(POINT["longitude"]).all()
table.write_csv(OUTPUT / "checklist.csv")
table.write_parquet(OUTPUT / "checklist.parquet")
with pl.Config(tbl_cols=-1, fmt_str_lengths=80):
    display(table.select("subcategory_name", "subcategory_value", "searched_latitude", "searched_longitude", "source_latitude", "source_longitude", "source_distance_m", "match_method").head(20))
print("Status:", site["status"], "Provider errors:", site["source_errors"], "Download errors:", site["evidence"]["errors"])

display(pl.from_dicts(site["public_coverage"]).select("source", "status", "records", "truncated", "detail"))

subcategory_name,subcategory_value,searched_latitude,searched_longitude,source_latitude,source_longitude,source_distance_m,match_method
str,str,f64,f64,f64,f64,f64,str
"""address""","""60 VASSAR ST""",42.359,-71.093,42.359925,-71.090924,199.19,"""point_intersection"""
"""municipality""","""CAMBRIDGE""",42.359,-71.093,null,null,null,"""municipality_containing_point"""
"""state""","""MA""",42.359,-71.093,null,null,null,"""municipality_containing_point"""
"""latitude""","""42.359""",42.359,-71.093,42.359,-71.093,0.0,"""caller_coordinates"""
"""longitude""","""-71.093""",42.359,-71.093,42.359,-71.093,0.0,"""caller_coordinates"""
…,…,…,…,…,…,…,…
"""parcel_attributes[1].TOWN_ID""","""49""",42.359,-71.093,42.359925,-71.090924,199.19,"""point_intersection"""
"""parcel_attributes[1].PROP_ID""","""52A-21""",42.359,-71.093,42.359925,-71.090924,199.19,"""point_intersection"""
"""parcel_attributes[1].BLDG_VAL""","""983213100""",42.359,-71.093,42.359925,-71.090924,199.19,"""point_intersection"""


Status: partial Provider errors: ['municipal_archives', 'land_records'] Download errors: []


source,status,records,truncated,detail
str,str,i64,bool,str
"""utilities""","""ok""",1,false,null
"""parcels""","""ok""",1,false,null
"""buildings""","""ok""",1,false,null
"""lber""","""ok""",14,false,null
"""flood""","""ok""",1,false,null
…,…,…,…,…
"""environmental""","""ok""",3,false,"""An empty result does not estab…"
"""activity_use_limits""","""ok""",7,false,"""An empty result does not estab…"
"""protected_habitat""","""no_data""",0,false,"""An empty result does not estab…"


## Test every public source independently

Sources needing an address or parcel automatically include the parcel lookup. Utilities always resolves the municipality. `no_data` means an empty published query, not proof that a physical feature or constraint is absent. `partial`, `blocked`, `unsupported` and `error` remain visible below.

In [4]:
provider_results = {}
for name in catalog["automated"]:
    payload = get("/site", {**POINT, "sources": name, "download_evidence": False}).json()
    result = payload["sources"][name]
    provider_results[name] = result
    print(name, result["status"], "records:", len(result["records"]), result.get("error", ""))
    if result["records"]:
        display(pl.from_dicts([{"record": index, **record["_location"]} for index, record in enumerate(result["records"][:10], 1)], infer_schema_length=None))
summary = pl.from_dicts([{"source": name, "status": result["status"], "records": len(result["records"]),
                        "retrieved_at_utc": result["retrieved_at_utc"], "detail": result.get("error") or result.get("detail") or result.get("limitations")} for name, result in provider_results.items()])
summary.write_csv(OUTPUT / "provider_checks.csv")
display(summary)
if REQUIRE_ALL_PROVIDERS:
    assert summary["status"].is_in(["ok", "no_data"]).all(), "Public coverage is incomplete; inspect provider detail above."


utilities ok records: 1 


record,searched_latitude,searched_longitude,source_latitude,source_longitude,source_distance_m,source_location_basis,match_method
i64,f64,f64,null,null,null,str,str
1,42.359,-71.093,null,null,null,"""Municipality-wide record; no s…","""municipality_containing_point"""


parcels ok records: 1 


record,searched_latitude,searched_longitude,source_latitude,source_longitude,source_distance_m,source_location_basis,match_method
i64,f64,f64,f64,f64,f64,str,str
1,42.359,-71.093,42.359925,-71.090924,199.19,"""Provider polygon centroid in W…","""point_intersection"""


buildings ok records: 1 


record,searched_latitude,searched_longitude,source_latitude,source_longitude,source_distance_m,source_location_basis,match_method
i64,f64,f64,f64,f64,f64,str,str
1,42.359,-71.093,42.360085,-71.091363,180.66,"""Provider polygon centroid in W…","""point_intersection"""


... (27 outputs left out)

record,searched_latitude,searched_longitude,source_latitude,source_longitude,source_distance_m,source_location_basis,match_method
i64,f64,f64,null,null,null,str,str
1,42.359,-71.093,null,null,null,"""Source location not supplied""","""municipality_context"""
2,42.359,-71.093,null,null,null,"""Source location not supplied""","""municipality_context"""
3,42.359,-71.093,null,null,null,"""Source location not supplied""","""municipality_context"""
4,42.359,-71.093,null,null,null,"""Source location not supplied""","""municipality_context"""
5,42.359,-71.093,null,null,null,"""Source location not supplied""","""municipality_context"""
6,42.359,-71.093,null,null,null,"""Source location not supplied""","""municipality_context"""
7,42.359,-71.093,null,null,null,"""Source location not supplied""","""municipality_context"""
8,42.359,-71.093,null,null,null,"""Source location not supplied""","""municipality_context"""
9,42.359,-71.093,null,null,null,"""Source location not supplied""","""municipality_context"""


tariffs ok records: 181 


record,searched_latitude,searched_longitude,source_latitude,source_longitude,source_distance_m,source_location_basis,match_method
i64,f64,f64,null,null,null,str,str
1,42.359,-71.093,null,null,null,"""Source location not supplied""","""utility_area_schedule_candidat…"
2,42.359,-71.093,null,null,null,"""Source location not supplied""","""utility_area_schedule_candidat…"
3,42.359,-71.093,null,null,null,"""Source location not supplied""","""utility_area_schedule_candidat…"
4,42.359,-71.093,null,null,null,"""Source location not supplied""","""utility_area_schedule_candidat…"
5,42.359,-71.093,null,null,null,"""Source location not supplied""","""utility_area_schedule_candidat…"
6,42.359,-71.093,null,null,null,"""Source location not supplied""","""utility_area_schedule_candidat…"
7,42.359,-71.093,null,null,null,"""Source location not supplied""","""utility_area_schedule_candidat…"
8,42.359,-71.093,null,null,null,"""Source location not supplied""","""utility_area_schedule_candidat…"
9,42.359,-71.093,null,null,null,"""Source location not supplied""","""utility_area_schedule_candidat…"


source,status,records,retrieved_at_utc,detail
str,str,i64,str,str
"""utilities""","""ok""",1,"""2026-11-07T20:59:57.159723+00:…",null
"""parcels""","""ok""",1,"""2026-11-07T20:59:57.460038+00:…",null
"""buildings""","""ok""",1,"""2026-11-07T20:59:57.835070+00:…",null
"""lber""","""ok""",14,"""2026-11-07T20:59:58.338205+00:…",null
"""flood""","""ok""",1,"""2026-11-07T20:59:59.363249+00:…",null
…,…,…,…,…
"""environmental""","""ok""",3,"""2026-11-07T21:00:06.255611+00:…","""An empty result does not estab…"
"""activity_use_limits""","""ok""",7,"""2026-11-07T21:00:06.732729+00:…","""An empty result does not estab…"
"""protected_habitat""","""no_data""",0,"""2026-11-07T21:00:07.156703+00:…","""An empty result does not estab…"


## Verify local evidence download URLs

`file_url` is relative to the API base URL. Each searched coordinate has its own `data/evidence/<latitude>_<longitude>/` folder, with downloaded files, immutable runs, a latest manifest and a readable index. These URLs serve only public evidence from that folder. Each response is checked against the saved SHA-256 hash. A selected PDF and Jupyter-relative file links are demonstrated in the short notebook.


Cached files preserve their original fetch timestamp. A manifest contains hashes, extraction limits and the whole parcel-envelope imagery tile coverage. All successful returned file URLs are fetched and their bytes checked below.

In [5]:
for asset in site["evidence"]["files"]:
    if asset["status"] == "ok":
        response = get(asset["file_url"])
        assert hashlib.sha256(response.content).hexdigest() == asset["sha256"]
        print(asset["source_title"], "→", BASE_URL + asset["file_url"])
if site["evidence"]["errors"]:
    display(pl.from_dicts([{ "source": x["source_url"], "error": x["error"] } for x in site["evidence"]["errors"]]))

for name, export in site["evidence"]["exports"].items():
    get(export["file_url"])
    print(name, "→", BASE_URL + export["file_url"])
get(site["evidence"]["index_url"])


MassGIS — electricity providers by municipality → http://127.0.0.1:8002/evidence/42.359000_-71.093000/runs/20261107T205954325603Z/utilities.json
MassGIS — Massachusetts assessor parcels → http://127.0.0.1:8002/evidence/42.359000_-71.093000/runs/20261107T205954325603Z/parcels.json
MassGIS — building structures and roof outlines → http://127.0.0.1:8002/evidence/42.359000_-71.093000/runs/20261107T205954325603Z/buildings.json
Massachusetts DOER — multi-year building energy disclosures → http://127.0.0.1:8002/evidence/42.359000_-71.093000/runs/20261107T205954325603Z/lber.json
FEMA — National Flood Hazard Layer → http://127.0.0.1:8002/evidence/42.359000_-71.093000/runs/20261107T205954325603Z/flood.json
City of Cambridge — Permit Finder → http://127.0.0.1:8002/evidence/42.359000_-71.093000/runs/20261107T205954325603Z/permits.json
City of Cambridge — detailed electrical permits → http://127.0.0.1:8002/evidence/42.359000_-71.093000/runs/20261107T205954325603Z/electrical_permits.json
City of Cam

MassCEC public PTS solar systems report → http://127.0.0.1:8002/evidence/42.359000_-71.093000/files/masscec-public-pts-solar-systems-report-b2435a2df0decef1aa94-9dfc1e8343092d5a.xlsx
Massachusetts official electric rate directory → http://127.0.0.1:8002/evidence/42.359000_-71.093000/files/public-page-8d474471f3466a9fd1a1.html
Utility-published Cambridge G-3 → http://127.0.0.1:8002/evidence/42.359000_-71.093000/files/utility-published-cambridge-g-3-171c8c22faf7f7a7bd3b-59e1a2c754ae48a2.pdf
Utility-published Cambridge G-3 — extracted text → http://127.0.0.1:8002/evidence/42.359000_-71.093000/files/utility-published-cambridge-g-3-171c8c22faf7f7a7bd3b-59e1a2c754ae48a2.txt
Utility-published Cambridge delivery summary → http://127.0.0.1:8002/evidence/42.359000_-71.093000/files/utility-published-cambridge-delivery-summary-3d0aee696ef9156db94b-9b55e7bb40291a32.pdf
Utility-published Cambridge delivery summary — extracted text → http://127.0.0.1:8002/evidence/42.359000_-71.093000/files/utility-p

Utility-published Referenced M.D.P.U. No. 1 rates — extracted text → http://127.0.0.1:8002/evidence/42.359000_-71.093000/files/utility-published-referenced-m-d-p-u-no-1-rates-1ff8da49c82e81dc320a-c8691d6937c77ccc.txt
MassGIS 2025 parcel-envelope imagery — tile 17/48478/39651 → http://127.0.0.1:8002/evidence/42.359000_-71.093000/files/massgis-2025-parcel-envelope-imagery-tile-17-48478-39651-f9f76c9c5072ab92ca1e-b4ec3fb2d10b2aeb.png
MassGIS 2025 parcel-envelope imagery — tile 17/48479/39651 → http://127.0.0.1:8002/evidence/42.359000_-71.093000/files/massgis-2025-parcel-envelope-imagery-tile-17-48479-39651-99820b2bccb7f52a9be5-1bdf149093cb3f11.jpg
MassGIS 2025 parcel-envelope imagery — tile 17/48480/39651 → http://127.0.0.1:8002/evidence/42.359000_-71.093000/files/massgis-2025-parcel-envelope-imagery-tile-17-48480-39651-4b86d7e9b545829d4859-46903d036af96a34.jpg
MassGIS 2025 parcel-envelope imagery — tile 17/48478/39652 → http://127.0.0.1:8002/evidence/42.359000_-71.093000/files/massgis-20

<Response [200]>

## Another facility and expected error responses

The same field names apply at a Fall River site. Automatic permit search currently covers Cambridge; the API explicitly returns a manual portal reference or blocked automation elsewhere. Unavailable fields stay null.


In [6]:
second = get("/site", {"latitude": 41.762125, "longitude": -71.100125, "sources": "parcels,buildings,lber", "download_evidence": False}).json()
assert [[f["name"] for f in c["fields"]] for c in second["checklist"]] == [[f["name"] for f in c["fields"]] for c in site["checklist"]]
for params in [{"longitude": -71}, {"latitude": 91, "longitude": -71}, {**POINT, "sources": "unknown"}, {"latitude": 40.7, "longitude": -74}]:
    get("/site", params, expected=422)
get("/evidence/nonexistent.pdf", expected=404)
get("/evidence/%2e%2e/%2e%2e/.env", expected=404)
http_checks = pl.from_dicts(checks)
http_checks.write_csv(OUTPUT / "http_checks.csv")
display(http_checks)
assert http_checks["passed"].all()
print("All HTTP, coordinate and evidence-link checks passed.")


path,url,expected_status,status,passed,seconds
str,str,i64,i64,bool,f64
"""/health""","""http://127.0.0.1:8002/health""",200,200,true,0.019
"""/openapi.json""","""http://127.0.0.1:8002/openapi.…",200,200,true,0.003
"""/sources""","""http://127.0.0.1:8002/sources""",200,200,true,0.001
"""/docs""","""http://127.0.0.1:8002/docs""",200,200,true,0.001
"""/redoc""","""http://127.0.0.1:8002/redoc""",200,200,true,0.001
…,…,…,…,…,…
"""/site""","""http://127.0.0.1:8002/site?lat…",422,422,true,0.003
"""/site""","""http://127.0.0.1:8002/site?lat…",422,422,true,0.003
"""/site""","""http://127.0.0.1:8002/site?lat…",422,422,true,0.13


All HTTP, coordinate and evidence-link checks passed.


Run `uv run python scripts/check_links.py` to verify external provider queries, reference pages and local files from the saved short-notebook response. The CSV report is written to `data/processed/link_checks.csv`. Live source and portal availability can change, so failures remain visible.
